# LISA posterior inference

Use trained checkpoints to estimate four gravitational-wave background parameters from one observation. **SBI-S** uses the separator’s signal and noise estimates. **SBI-R** uses the measured spectra directly.

Follow the [README](../README.md) to train the models. Select the same Python environment as your Jupyter kernel. The quick training run is enough to check the workflow. If needed, install JupyterLab with `python -m pip install jupyterlab` in that environment and launch it with `jupyter lab`.

For SBI-S, use the same separator checkpoints that generated its training data. The SBI checkpoint includes the fitted standardisation and PCA. Load only trusted checkpoint files.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from lisa_gwb.sbi import sample, summary
from lisa_gwb.simulation import (
    MissionConfig,
    SBI_PARAMETER_NAMES,
    SignalPriors,
    draw_sbi_parameters,
    make_training_batch,
    multichannel_f_bin,
    sample_gal_modulation,
    sbi_parameters_to_event,
)

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "training" / "sbi_paper.json").is_file()), None)
if REPO is None:
    raise RuntimeError("Start this notebook from the repository or its tutorials directory.")

VARIANT = "sbi-s"  # Set to "sbi-r" for inference on measured spectra.
RESULTS = REPO / "results"  # Use REPO / "results" / "smoke" after the quick run.
SPECTRUM_FILE = None  # Or Path("my_observation.npy"), relative to the repository.
INPUT_UNITS = "log10"  # Choose "linear" for a file of PSD values in Hz^-1.
SEPARATOR_SEEDS = (20260922, 20260923, 20260924)
N_DRAWS = 2000

if VARIANT not in {"sbi-s", "sbi-r"}:
    raise ValueError("VARIANT must be 'sbi-s' or 'sbi-r'.")
if INPUT_UNITS not in {"log10", "linear"}:
    raise ValueError("INPUT_UNITS must be 'log10' or 'linear'.")
if N_DRAWS < 1:
    raise ValueError("N_DRAWS must be positive.")

torch.set_num_threads(2)

## 1. Prepare one observation

By default, the simulator draws a fresh observation with known parameters. To supply your own input, set `SPECTRUM_FILE` to a NumPy `.npy` array of shape **(3, 512)** in A, E, T order. Use the same four-year, 126-segment averaging and coarse-graining as in training. `frequencies` below gives the channel-specific bin centres. Linear PSD input is converted to physical log10 PSD here. The separator applies its own normalisation.

In [ ]:
mission = MissionConfig()
channels = ("A", "E", "T")
frequencies = multichannel_f_bin(mission, 512, channels).reshape(3, 512)
truth = None

if SPECTRUM_FILE is None:
    rng = np.random.default_rng(314159)
    priors = SignalPriors(gal_modulation=True)
    truth = draw_sbi_parameters(rng)
    event = sbi_parameters_to_event(truth, priors)
    event.update(sample_gal_modulation(rng, priors))
    measured = make_training_batch(
        1, 1, 512, mission, priors, rng, channels=channels,
        fixed_params=[event], normalize=False, n_avg=mission.Nd, draw="gamma",
    )[0][0]
else:
    measured = np.load(REPO / SPECTRUM_FILE, allow_pickle=False)
    if not isinstance(measured, np.ndarray) or measured.shape != (3, 512):
        raise ValueError("Use a (3, 512) .npy array in A, E, T order.")
    if (not np.issubdtype(measured.dtype, np.number) or not np.isrealobj(measured)
            or not np.isfinite(measured).all()):
        raise ValueError("Every PSD value must be real and finite.")
    if INPUT_UNITS == "linear":
        if np.any(measured <= 0):
            raise ValueError("Linear PSD values must be positive.")
        measured = np.log10(measured)

observed = {"x": measured[None].astype(np.float32)}

## 2. Prepare the network input

For SBI-S, load the three separator seeds and average their predictions in log power. The posterior receives predicted gravitational-wave power in A/E and predicted noise in A/E/T. For SBI-R, pass the measured spectra directly. To see just the separator output, run through this step.

In [ ]:
kind = "primary" if VARIANT == "sbi-s" else "control"

if VARIANT == "sbi-s":
    from lisa_gwb.separator import load_separators, separate_spectra

    paths = [RESULTS / "prod" / f"seed{seed}" / "best.pt" for seed in SEPARATOR_SEEDS]
    models = load_separators(paths, mission)
    observed["s_hat"], observed["n_hat"] = separate_spectra(models, observed["x"])

spectra = summary(kind, observed)
print(f"{VARIANT} input shape before PCA: {spectra.shape}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3), layout="constrained")
for i, (channel, ax) in enumerate(zip(channels, axes)):
    ax.semilogx(frequencies[i], measured[i], label="Measured")
    if VARIANT == "sbi-s":
        ax.semilogx(frequencies[i], observed["n_hat"][0, i], label="Noise estimate")
        ax.semilogx(frequencies[i], observed["s_hat"][0, i], label="GW estimate")
    ax.set(title=channel, xlabel="Frequency [Hz]")
axes[0].set_ylabel("log10 PSD [Hz⁻¹]")
axes[-1].legend(fontsize=8)
plt.show()

## 3. Draw posterior samples

`draws` has one row per draw and columns in `SBI_PARAMETER_NAMES` order. All four parameters are in base-10 log units. The printed summary gives the median and central 68% credible interval. For a simulated observation, dashed lines mark the injected parameters.

In [ ]:
posterior_path = RESULTS / VARIANT / "posterior.pt"
if not posterior_path.is_file():
    raise FileNotFoundError(
        f"{posterior_path} is missing. Train {VARIANT} as shown in the README "
        "or point RESULTS to the directory containing your checkpoints."
    )
checkpoint = torch.load(posterior_path, map_location="cpu", weights_only=False)
if checkpoint["kind"] != kind:
    raise ValueError(f"{posterior_path} is incompatible with {VARIANT}.")
draws = sample(checkpoint["posterior"], spectra, n=N_DRAWS)[:, 0, :]
low, median, high = np.quantile(draws, [0.16, 0.50, 0.84], axis=0)

for name, lo, mid, hi in zip(SBI_PARAMETER_NAMES, low, median, high):
    print(f"{name:20} {mid:7.3f}  (68% interval {lo:.3f} to {hi:.3f})")

fig, axes = plt.subplots(2, 2, figsize=(8, 5), layout="constrained")
for i, (name, ax) in enumerate(zip(SBI_PARAMETER_NAMES, axes.flat)):
    ax.hist(draws[:, i], bins=40, density=True)
    if truth is not None:
        ax.axvline(truth[i], ls="--", c="black", label="Injected")
    ax.set(xlabel=name, ylabel="Posterior density")
if truth is not None:
    axes[0, 0].legend()
plt.show()